# Qwen-Image-2.1: one text-to-image sample with the psoriasis LoRA, one edit sample

Part 1 loads the text-to-image LoRA used in the Figure 3 sweep (checkpoint 1500) and generates one image.
Part 2 drops the LoRA and runs the sweep's edit arm: untuned Qwen, a real psoriasis image as reference, one of the 10 varied prompts.
Models load from `HF_HOME` (set it to the RAM copy, e.g. `/dev/shm/hf`, before starting Jupyter); the LoRA and the pool CSV are read from `/workspace`.

In [ ]:
import os, torch
from diffusers import QwenImage21Pipeline

GPU = 0
LABEL = "psoriasis"
LORA = f"/workspace/synderm/qwen_lora/t2i/{LABEL}/r16-lr0.0001-512px-2000steps/checkpoint-1500/pytorch_lora_weights.safetensors"
PROMPT = f"An image of {LABEL}, a skin disease"   # the paper's prompt, as in the sweep
SEED = 0
print("HF_HOME =", os.environ.get("HF_HOME"), "| LoRA exists:", os.path.exists(LORA))

In [ ]:
pipe = QwenImage21Pipeline.from_pretrained("Qwen/Qwen-Image-2.1", dtype=torch.bfloat16).to(f"cuda:{GPU}")
pipe.load_lora_weights(os.path.dirname(LORA), weight_name=os.path.basename(LORA))
pipe.set_progress_bar_config(disable=True)

In [ ]:
image = pipe(prompt=PROMPT, height=512, width=512, num_inference_steps=40,
             generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
image

In [ ]:
image.save(f"/workspace/synderm/qwen_lora/sample_{LABEL}_ckpt1500_seed{SEED}.png")

## Edit mode (untuned Qwen, as in the sweep's edit arm)

In [ ]:
import pandas as pd
from PIL import Image

IMAGES_DIR = "/root/data/fitzpatrick17k/finalfitz17k"
pool9 = pd.read_csv("/workspace/synderm/qwen_lora/generations/pool9.csv")
ref_md5 = pool9[pool9.label == LABEL].md5hash.iloc[0]
reference = Image.open(f"{IMAGES_DIR}/{ref_md5}.jpg").convert("RGB")
# prompt as in qwen_lora_generate.py --method edit --prompts varied, slot 0
EDIT_PROMPT = (f"A clinical photograph of the same skin disease, {LABEL}, keeping the characteristic appearance "
               "of the lesions and the same skin tone as the reference, on a different patient. "
               "No text, watermarks or annotations. Close-up of a single lesion on the forearm, exam-room lighting.")
pipe.unload_lora_weights()
edit = pipe(prompt=EDIT_PROMPT, image=reference, output_resolution=512, num_inference_steps=40,
            generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
pair = Image.new("RGB", (1024, 512)); pair.paste(reference.resize((512, 512)), (0, 0)); pair.paste(edit.resize((512, 512)), (512, 0))
pair   # reference on the left, edit on the right

In [ ]:
edit.save(f"/workspace/synderm/qwen_lora/sample_{LABEL}_edit_untuned_seed{SEED}.png")